In [1]:
import pandas as pd

In [2]:
# Loading Dataset
df = pd.read_csv('311_sample_10k.csv')

In [ ]:
# What we are working with 
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nAny weird values in date columns?")
print(df['Created Date'].head(10))

In [ ]:
# Clean column names (remove spaces, special chars, make PostgreSQL-friendly)
df.columns = (df.columns
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('formerly_', '')
)
print("Cleaned column names:")
print(df.columns.tolist())

In [ ]:
# Convert dates (the format is consistent: MM/DD/YYYY HH:MM:SS AM/PM)
df['created_date'] = pd.to_datetime(df['created_date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df['closed_date'] = pd.to_datetime(df['closed_date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

# Check conversion success
print(f"Created date conversion failures: {df['created_date'].isna().sum()} rows")
print(f"Closed date conversion failures: {df['closed_date'].isna().sum()} rows")
print(f"\nSample of converted dates:")
print(df[['created_date', 'closed_date']].head())

In [ ]:
# Keep only rows where created_date converted successfully
df_clean = df.dropna(subset=['created_date'])

print(f"Original rows: {len(df)}")
print(f"After dropping invalid dates: {len(df_clean)}")

In [ ]:
# Choose columns that will make sense in your portfolio projects
portfolio_columns = [
    'unique_key',
    'created_date', 
    'closed_date',
    'agency',
    'agency_name',
    'problem_complaint_type',      # ← correct name
    'problem_detail_descriptor',       # this was 'problem_detail'
    'location_type',
    'incident_zip',
    'status',
    'borough',
    'latitude',
    'longitude'
]


# Keep only columns that exist in your dataframe
existing_columns = [col for col in portfolio_columns if col in df_clean.columns]
df_portfolio = df_clean[existing_columns]

print(f"Keeping {len(existing_columns)} columns: {existing_columns}")
print(f"\nShape after column selection: {df_portfolio.shape}")
df_portfolio.head()

In [19]:
# Create a COPY (this eliminates the warning)
df_portfolio = df_clean[portfolio_columns].copy()
# Rename to cleaner names for PostgreSQL
df_portfolio.rename(columns={
    'problem_complaint_type': 'complaint_type',
    'problem_detail_descriptor': 'descriptor'
}, inplace=True)

In [20]:
# Convert numeric columns properly
df_portfolio['unique_key'] = df_portfolio['unique_key'].astype('int64')
df_portfolio['incident_zip'] = df_portfolio['incident_zip'].fillna(0).astype('int64')
df_portfolio['latitude'] = pd.to_numeric(df_portfolio['latitude'], errors='coerce')
df_portfolio['longitude'] = pd.to_numeric(df_portfolio['longitude'], errors='coerce')

# Preview final clean data
print("Final clean data types:")
print(df_portfolio.dtypes)
print("\nSample rows:")
df_portfolio.head(10)

Final clean data types:
unique_key                 int64
created_date      datetime64[ns]
closed_date       datetime64[ns]
agency                    object
agency_name               object
complaint_type            object
descriptor                object
location_type             object
incident_zip               int64
status                    object
borough                   object
latitude                 float64
longitude                float64
dtype: object

Sample rows:


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,status,borough,latitude,longitude
0,26037396,2013-07-24 15:43:31,2013-07-31 10:08:31,DOF,Property Exec Office,DOF Property - Owner Issue,Billing Address Incorrect,Property Address,11218,Closed,BROOKLYN,NaN,NaN
1,26045764,2013-07-24 10:36:45,2013-07-24 15:02:57,DOF,Correspondence Unit,DOF Property - Request Copy,Copy of Statement,Property Address,11372,Closed,QUEENS,NaN,NaN
2,26045943,2013-08-01 09:50:08,2013-08-06 11:53:12,DOF,Property Exec Office,DOF Property - Owner Issue,Billing Address Incorrect,Property Address,11691,Closed,QUEENS,NaN,NaN
3,26040205,2013-07-31 15:01:00,2013-07-31 15:01:00,DSNY,Department of Sanitation,Literature Request,LIT Literature Request,Other,11235,Closed,BROOKLYN,NaN,NaN
4,26047611,2013-08-01 14:01:00,2013-08-01 14:01:00,DSNY,Department of Sanitation,Literature Request,MERCHANT CLEAN COMM. FLYER,Other,10464,Closed,BRONX,NaN,NaN
5,26039427,2013-07-31 17:04:42,2013-08-01 10:05:45,DOF,Adjudication - Hearing by Mail,DOF Parking - Request Status,Status of Hearing,NaN,0,Closed,Unspecified,NaN,NaN
6,26041273,2013-07-31 15:48:12,2013-08-13 08:28:03,DOF,Refunds and Adjustments,DOF Parking - Payment Issue,Finance Business Center - Not Reflected,NaN,0,Closed,Unspecified,NaN,NaN
7,26038297,2013-07-31 14:53:00,2013-07-31 14:53:00,DSNY,Department of Sanitation,Literature Request,BLUE RECYCLING DECAL,Other,10471,Closed,BRONX,NaN,NaN
8,26038282,2013-07-31 08:03:00,2013-07-31 12:00:00,DSNY,Department of Sanitation,Literature Request,LIT Literature Request,Other,11207,Closed,BROOKLYN,NaN,NaN
9,26037290,2013-07-30 15:05:00,2013-07-30 15:05:00,DSNY,Department of Sanitation,Literature Request,LIT Literature Request,Other,11363,Closed,QUEENS,NaN,NaN


In [21]:
# Save to CSV (PostgreSQL will love this)
df_portfolio.to_csv('311_clean_portfolio.csv', index=False)

print("✅ Exported to '311_clean_portfolio.csv'")
print(f"Total rows: {len(df_portfolio)}")
print(f"Total columns: {len(df_portfolio.columns)}")
print("\nFile ready for PostgreSQL import!")

✅ Exported to '311_clean_portfolio.csv'
Total rows: 10000
Total columns: 13

File ready for PostgreSQL import!
